# Ultrasound — dataset preparation

Preparation of BUS-BRA development data and the BrEaST held-out cohort, including ROI generation and patient-level partition checks.


In [ ]:
!pip -q install openpyxl pillow opencv-python-headless scikit-learn tqdm beautifulsoup4 requests

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import zipfile
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

MOUNT_DRIVE = True
if IN_COLAB and MOUNT_DRIVE:
    drive.mount('/content/drive')

BASE_ROOT = Path('/content/drive/MyDrive' if MOUNT_DRIVE else '/content')
PROJECT_ROOT = BASE_ROOT / 'TRACKC_ULTRASOUND'
RAW_ROOT = PROJECT_ROOT / 'raw'
PREPARED_ROOT = PROJECT_ROOT / 'ROI_US_Crops_256_v1'
DOWNLOAD_ROOT = RAW_ROOT / 'downloads'
EXTRACT_ROOT = RAW_ROOT / 'extracted'

for folder in [PROJECT_ROOT, RAW_ROOT, PREPARED_ROOT, DOWNLOAD_ROOT, EXTRACT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

BUSBRA_RECORD_ID = 8231412
BUSBRA_ZENODO_API = f'https://zenodo.org/api/records/{BUSBRA_RECORD_ID}'

BREAST_ATTACHMENT_PAGE = (
    'https://www.cancerimagingarchive.net/tcia-downloads/'
    'breast-lesions-usg-da-rad/'
    'breast-lesions_usg-images_and_masks-dec-15-2023/'
)
BREAST_XLSX_URL = (
    'https://www.cancerimagingarchive.net/wp-content/uploads/'
    'BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx'
)

BREAST_ZIP_CANDIDATES = [
    'https://www.cancerimagingarchive.net/wp-content/uploads/'
    'BrEaST-Lesions_USG-images_and_masks-Dec-15-2023.zip',
    'https://www.cancerimagingarchive.net/wp-content/uploads/'
    'BrEaST-Lesions-USG-images_and_masks-Dec-15-2023.zip',
    'https://www.cancerimagingarchive.net/wp-content/uploads/'
    'Breast-Lesions_USG-images_and_masks-Dec-15-2023.zip',
]

CROP_SIZE = 256
CONTEXT_FACTOR = 1.50
RANDOM_SEED = 2026
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
ALLOW_FILENAME_PATIENT_FALLBACK = False
RUN_DOWNLOAD = True
RUN_PREPARATION = True
CREATE_FULL_ZIP = True

print('Project root:', PROJECT_ROOT)

In [ ]:
def md5_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def download_stream(url: str, destination: Path, timeout: int = 120) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 1024:
        print('Already present:', destination)
        return destination

    temporary = destination.with_suffix(destination.suffix + '.part')
    with requests.get(url, stream=True, timeout=timeout, allow_redirects=True) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length', 0))
        with temporary.open('wb') as handle, tqdm(
            total=total, unit='B', unit_scale=True, desc=destination.name
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
                    progress.update(len(chunk))
    temporary.replace(destination)
    return destination

def is_valid_zip(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 1000 and zipfile.is_zipfile(path)

def download_busbra() -> Path:
    metadata = requests.get(BUSBRA_ZENODO_API, timeout=60).json()
    files = metadata.get('files', [])
    candidate = next(
        (item for item in files if item.get('key', '').lower() == 'busbra.zip'),
        None,
    )
    if candidate is None:
        raise RuntimeError('BUSBRA.zip was not found in the Zenodo record.')

    links = candidate.get('links', {})
    url = links.get('content') or links.get('self')
    if not url:
        raise RuntimeError('Zenodo file URL is missing.')

    output = DOWNLOAD_ROOT / 'BUSBRA.zip'
    download_stream(url, output)

    checksum = candidate.get('checksum', '')
    if checksum.startswith('md5:'):
        expected = checksum.split(':', 1)[1]
        observed = md5_file(output)
        if observed != expected:
            raise RuntimeError(f'BUS-BRA MD5 mismatch: {observed} != {expected}')
    if not is_valid_zip(output):
        raise RuntimeError('BUS-BRA download is not a valid ZIP.')
    return output

def discover_breast_zip_url() -> Optional[str]:
    for url in BREAST_ZIP_CANDIDATES:
        try:
            response = requests.get(url, stream=True, timeout=30, allow_redirects=True)
            content_type = response.headers.get('content-type', '').lower()
            first = next(response.iter_content(chunk_size=8), b'')
            response.close()
            if response.status_code == 200 and (
                first.startswith(b'PK') or 'zip' in content_type
            ):
                return url
        except Exception:
            pass

    media_api = (
        'https://www.cancerimagingarchive.net/wp-json/wp/v2/media'
        '?search=BrEaST-Lesions_USG-images_and_masks&per_page=100'
    )
    try:
        items = requests.get(media_api, timeout=60).json()
        for item in items:
            source_url = str(item.get('source_url', ''))
            if source_url.lower().endswith('.zip'):
                return source_url
    except Exception:
        pass

    try:
        html = requests.get(BREAST_ATTACHMENT_PAGE, timeout=60).text
        soup = BeautifulSoup(html, 'html.parser')
        urls = []
        for tag in soup.find_all(['a', 'meta']):
            value = tag.get('href') or tag.get('content')
            if value and '.zip' in value.lower():
                urls.append(value)
        for match in re.findall(r'https?://[^"\'\s]+\.zip(?:\?[^"\'\s]*)?', html):
            urls.append(match)
        if urls:
            return urls[0]
    except Exception:
        pass
    return None

def download_breast() -> Tuple[Path, Path]:
    zip_path = DOWNLOAD_ROOT / 'BrEaST-images-and-masks.zip'
    xlsx_path = DOWNLOAD_ROOT / 'BrEaST-clinical-data.xlsx'

    if not is_valid_zip(zip_path):
        zip_url = discover_breast_zip_url()
        if zip_url is None:
            raise RuntimeError(
                'Automatic BrEaST ZIP discovery failed. Download the official images/masks ZIP '
                'from the TCIA BREAST-LESIONS-USG page and place it at: '
                f'{zip_path}'
            )
        print('BrEaST ZIP URL:', zip_url)
        download_stream(zip_url, zip_path)

    if not is_valid_zip(zip_path):
        raise RuntimeError('BrEaST images/masks archive is not a valid ZIP.')

    download_stream(BREAST_XLSX_URL, xlsx_path)
    if xlsx_path.stat().st_size < 1000:
        raise RuntimeError('BrEaST clinical XLSX appears incomplete.')
    return zip_path, xlsx_path

if RUN_DOWNLOAD:
    BUSBRA_ZIP = download_busbra()
    BREAST_ZIP, BREAST_XLSX = download_breast()
    print(BUSBRA_ZIP)
    print(BREAST_ZIP)
    print(BREAST_XLSX)

In [ ]:
def safe_extract(zip_path: Path, destination: Path) -> Path:
    marker = destination / '.extracted_ok'
    if marker.exists():
        return destination

    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        root = destination.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    marker.write_text('ok', encoding='utf-8')
    return destination

BUSBRA_EXTRACTED = safe_extract(
    DOWNLOAD_ROOT / 'BUSBRA.zip',
    EXTRACT_ROOT / 'BUSBRA',
)
BREAST_EXTRACTED = safe_extract(
    DOWNLOAD_ROOT / 'BrEaST-images-and-masks.zip',
    EXTRACT_ROOT / 'BREAST',
)

print('BUS-BRA files:', sum(1 for _ in BUSBRA_EXTRACTED.rglob('*')))
print('BrEaST files:', sum(1 for _ in BREAST_EXTRACTED.rglob('*')))

In [ ]:
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def image_files(root: Path) -> List[Path]:
    return sorted(
        path for path in root.rglob('*')
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

def find_named_dir(root: Path, names: Iterable[str]) -> Optional[Path]:
    normalized = {name.lower() for name in names}
    matches = [
        path for path in root.rglob('*')
        if path.is_dir() and path.name.lower() in normalized
    ]
    return matches[0] if matches else None

def read_gray(path: Path) -> np.ndarray:
    with Image.open(path) as image:
        if image.mode in {'RGBA', 'LA'}:
            rgba = image.convert('RGBA')
            background = Image.new('RGBA', rgba.size, (0, 0, 0, 255))
            image = Image.alpha_composite(background, rgba).convert('L')
        else:
            image = image.convert('L')
        array = np.asarray(image)
    return array

def read_binary_mask(path: Path) -> np.ndarray:
    array = read_gray(path)
    return (array > 0).astype(np.uint8)

def normalize_ultrasound(image: np.ndarray) -> np.ndarray:
    image = image.astype(np.float32)
    finite = np.isfinite(image)
    if not finite.any():
        raise ValueError('Image contains no finite pixels.')
    values = image[finite]
    nonzero = values[values > 0]
    reference = nonzero if nonzero.size >= 100 else values
    lo, hi = np.percentile(reference, [0.5, 99.5])
    if hi <= lo:
        lo, hi = float(values.min()), float(values.max())
    if hi <= lo:
        return np.zeros_like(image, dtype=np.float32)
    return np.clip((image - lo) / (hi - lo), 0, 1).astype(np.float32)

def mask_bbox(mask: np.ndarray) -> Tuple[int, int, int, int]:
    ys, xs = np.where(mask > 0)
    if xs.size == 0:
        raise ValueError('Empty mask.')
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1

def crop_square_with_context(
    image: np.ndarray,
    mask: np.ndarray,
    factor: float = CONTEXT_FACTOR,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, int]]:
    x0, y0, x1, y1 = mask_bbox(mask)
    bbox_w, bbox_h = x1 - x0, y1 - y0
    side = max(2, int(np.ceil(max(bbox_w, bbox_h) * factor)))
    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2

    left = int(np.floor(cx - side / 2))
    top = int(np.floor(cy - side / 2))
    right = left + side
    bottom = top + side

    h, w = image.shape[:2]
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - w)
    pad_bottom = max(0, bottom - h)

    image_pad = np.pad(
        image,
        ((pad_top, pad_bottom), (pad_left, pad_right)),
        mode='constant',
        constant_values=0,
    )
    mask_pad = np.pad(
        mask,
        ((pad_top, pad_bottom), (pad_left, pad_right)),
        mode='constant',
        constant_values=0,
    )

    left += pad_left
    right += pad_left
    top += pad_top
    bottom += pad_top

    image_crop = image_pad[top:bottom, left:right]
    mask_crop = mask_pad[top:bottom, left:right]

    meta = {
        'bbox_x0': x0, 'bbox_y0': y0, 'bbox_x1': x1, 'bbox_y1': y1,
        'bbox_w': bbox_w, 'bbox_h': bbox_h,
        'crop_size_native': side,
        'crop_touches_border': int(
            pad_left > 0 or pad_top > 0 or pad_right > 0 or pad_bottom > 0
        ),
    }
    return image_crop, mask_crop, meta

def resize_pair(image: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    image_resized = cv2.resize(
        image, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_LINEAR
    )
    mask_resized = cv2.resize(
        mask, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_NEAREST
    )
    return image_resized.astype(np.float32), (mask_resized > 0).astype(np.uint8)

def array_hash(array: np.ndarray) -> str:
    contiguous = np.ascontiguousarray(array)
    return hashlib.sha256(contiguous.tobytes()).hexdigest()

def normalize_col(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', str(name).lower()).strip('_')

In [ ]:

EXPECTED_BUSBRA_IMAGES = 1875
EXPECTED_BUSBRA_PATIENTS = 1064

def choose_column(
    columns: Iterable[str],
    aliases: Iterable[str],
) -> Optional[str]:
    mapping = {
        normalize_col(column): column
        for column in columns
    }

    for alias in aliases:
        key = normalize_col(alias)
        if key in mapping:
            return mapping[key]

    for alias in aliases:
        key = normalize_col(alias)
        for normalized, original in mapping.items():
            if key in normalized or normalized in key:
                return original

    return None

def locate_busbra_dirs(
    root: Path,
) -> Tuple[Path, Path]:
    images_dir = find_named_dir(
        root,
        ['Images', 'Image', 'images'],
    )
    masks_dir = find_named_dir(
        root,
        ['Masks', 'Mask', 'masks'],
    )

    if images_dir is None or masks_dir is None:
        available_dirs = sorted(
            str(path.relative_to(root))
            for path in root.rglob('*')
            if path.is_dir()
        )[:100]

        raise RuntimeError(
            'Could not locate BUS-BRA Images/Masks directories. '
            f'Available directories include: {available_dirs}'
        )

    print('BUS-BRA Images directory:', images_dir)
    print('BUS-BRA Masks directory:', masks_dir)
    return images_dir, masks_dir

def load_busbra_metadata(
    root: Path,
) -> pd.DataFrame:
    csv_paths = sorted(root.rglob('*.csv'))

    if not csv_paths:
        raise RuntimeError(
            'No BUS-BRA CSV metadata files were found.'
        )

    candidates = []

    for path in csv_paths:
        frames = []

        for separator in [',', ';']:
            try:
                frame = pd.read_csv(
                    path,
                    sep=separator,
                    dtype=str,
                    keep_default_na=False,
                    encoding='utf-8-sig',
                )
                if len(frame.columns) > 1:
                    frames.append(frame)
            except Exception:
                pass

        if not frames:
            continue

        frame = max(
            frames,
            key=lambda item: len(item.columns),
        )

        normalized_columns = {
            normalize_col(column)
            for column in frame.columns
        }

        score = 0
        score += 100 if path.name.lower() == 'bus_data.csv' else 0
        score += 20 if 'id' in normalized_columns else 0
        score += 20 if 'case' in normalized_columns else 0
        score += 15 if 'side' in normalized_columns else 0
        score += 10 if 'pathology' in normalized_columns else 0
        score += 10 if 'birads' in normalized_columns else 0
        score += 5 if 'device' in normalized_columns else 0

        candidates.append(
            (
                score,
                len(frame),
                path,
                frame,
            )
        )

    if not candidates:
        raise RuntimeError(
            'BUS-BRA CSV files could not be read.'
        )

    candidates.sort(
        key=lambda item: (item[0], item[1]),
        reverse=True,
    )

    score, rows, path, frame = candidates[0]
    frame.columns = [
        str(column).strip()
        for column in frame.columns
    ]

    print('Selected BUS-BRA metadata:', path)
    print('Metadata rows:', len(frame))
    print('Columns:', list(frame.columns))

    return frame

def clean_identifier(value) -> str:
    if value is None:
        return ''

    value = str(value).strip()

    if not value or value.lower() in {'nan', 'none'}:
        return ''

    if re.fullmatch(r'\d+\.0', value):
        value = value[:-2]

    return value

def normalize_busbra_id(value) -> str:
    """
    Canonicalise l'ID sans perdre les identifiants alphanumériques.
    Les IDs purement numériques sont convertis en entiers textuels afin que
    0007, 7 et 7.0 correspondent.
    """
    value = clean_identifier(value).lower()
    value = Path(value.replace('\\', '/')).stem

    value = re.sub(
        r'^(bus|image|img|mask|seg|gt)[_\-\s]*',
        '',
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r'[_\-\s]+(left|right|lt|rt|l|r)$',
        '',
        value,
        flags=re.IGNORECASE,
    )

    value = value.strip('_- ')

    if re.fullmatch(r'\d+', value):
        return str(int(value))

    return re.sub(r'[^a-z0-9]+', '', value)

def normalize_busbra_side(value) -> str:
    value = clean_identifier(value).lower()
    value = re.sub(r'[^a-z0-9]+', '', value)

    side_map = {
        'l': 'l',
        'left': 'l',
        'lt': 'l',
        'esq': 'l',
        'esquerda': 'l',
        'r': 'r',
        'right': 'r',
        'rt': 'r',
        'dir': 'r',
        'direita': 'r',
    }

    if value in side_map:
        return side_map[value]

    # Le dataset peut utiliser d'autres codes : on les garde plutôt que de
    return value

def parse_busbra_filename(
    path: Path,
    expected_kind: str,
) -> Tuple[str, str]:
    """
    Analyse notamment :
      bus_0007-l.png
      bus-7-R.png
      mask_0007-l.png
      mask-7-right.png
    """
    stem = path.stem.strip().lower()

    if expected_kind == 'image':
        stem = re.sub(
            r'^(bus|image|img)[_\-\s]*',
            '',
            stem,
            flags=re.IGNORECASE,
        )
    else:
        stem = re.sub(
            r'^(mask|segmentation|seg|gt)[_\-\s]*',
            '',
            stem,
            flags=re.IGNORECASE,
        )

    match = re.match(
        r'^(.*?)[_\-\s]+(left|right|lt|rt|l|r)$',
        stem,
        flags=re.IGNORECASE,
    )

    if match:
        raw_id = match.group(1)
        raw_side = match.group(2)
    else:
        compact_match = re.match(
            r'^(.*?)(left|right|lt|rt|l|r)$',
            stem,
            flags=re.IGNORECASE,
        )
        if compact_match:
            raw_id = compact_match.group(1)
            raw_side = compact_match.group(2)
        else:
            raw_id = stem
            raw_side = ''

    return (
        normalize_busbra_id(raw_id),
        normalize_busbra_side(raw_side),
    )

def build_busbra_asset_index(
    paths: List[Path],
    expected_kind: str,
) -> Tuple[
    Dict[Tuple[str, str], List[Path]],
    Dict[str, List[Path]],
    pd.DataFrame,
]:
    """
    Deux index :
      1) clé stricte (ID, Side) ;
      2) clé ID seule, utilisable seulement si elle est unique.
    """
    by_id_side = {}
    by_id = {}
    parsed_rows = []

    for path in paths:
        normalized_id, normalized_side = parse_busbra_filename(
            path,
            expected_kind=expected_kind,
        )

        by_id_side.setdefault(
            (normalized_id, normalized_side),
            [],
        ).append(path)

        by_id.setdefault(
            normalized_id,
            [],
        ).append(path)

        parsed_rows.append(
            {
                'kind': expected_kind,
                'filename': path.name,
                'parsed_id': normalized_id,
                'parsed_side': normalized_side,
            }
        )

    return (
        by_id_side,
        by_id,
        pd.DataFrame(parsed_rows),
    )

def lookup_busbra_asset(
    by_id_side: Dict[Tuple[str, str], List[Path]],
    by_id: Dict[str, List[Path]],
    normalized_id: str,
    normalized_side: str,
) -> Tuple[Optional[Path], str]:
    """
    Priorité à (ID, Side). Le fallback ID seul n'est accepté que s'il produit
    exactement un fichier, afin d'éviter tout appariement ambigu.
    """
    strict_matches = by_id_side.get(
        (normalized_id, normalized_side),
        [],
    )

    if len(strict_matches) == 1:
        return strict_matches[0], 'id_side'

    if len(strict_matches) > 1:
        return None, 'ambiguous_id_side'

    id_matches = by_id.get(
        normalized_id,
        [],
    )

    if len(id_matches) == 1:
        return id_matches[0], 'unique_id_fallback'

    if len(id_matches) > 1:
        return None, 'ambiguous_id_only'

    return None, 'not_found'

def build_busbra_records(
    root: Path,
) -> pd.DataFrame:
    images_dir, masks_dir = locate_busbra_dirs(root)

    images = image_files(images_dir)
    masks = image_files(masks_dir)

    print('BUS-BRA image files found:', len(images))
    print('BUS-BRA mask files found:', len(masks))

    print(
        'First BUS-BRA image filenames:',
        [path.name for path in images[:12]],
    )
    print(
        'First BUS-BRA mask filenames:',
        [path.name for path in masks[:12]],
    )

    if not images or not masks:
        raise RuntimeError(
            'BUS-BRA Images or Masks directory is empty.'
        )

    (
        image_by_id_side,
        image_by_id,
        parsed_images,
    ) = build_busbra_asset_index(
        images,
        expected_kind='image',
    )

    (
        mask_by_id_side,
        mask_by_id,
        parsed_masks,
    ) = build_busbra_asset_index(
        masks,
        expected_kind='mask',
    )

    parsed_assets = pd.concat(
        [parsed_images, parsed_masks],
        ignore_index=True,
    )
    parsed_assets_path = (
        PROJECT_ROOT
        / 'busbra_parsed_asset_filenames.csv'
    )
    parsed_assets.to_csv(
        parsed_assets_path,
        index=False,
    )

    metadata = load_busbra_metadata(root)

    image_col = choose_column(
        metadata.columns,
        ['ID', 'image_id', 'image_identifier'],
    )
    patient_col = choose_column(
        metadata.columns,
        ['Case', 'case_id', 'patient_id', 'patient'],
    )
    side_col = choose_column(
        metadata.columns,
        ['Side', 'breast_side', 'laterality'],
    )
    histology_col = choose_column(
        metadata.columns,
        ['Histology', 'histological_diagnosis'],
    )
    pathology_col = choose_column(
        metadata.columns,
        ['Pathology', 'classification', 'diagnosis'],
    )
    birads_col = choose_column(
        metadata.columns,
        ['BIRADS', 'bi_rads', 'birads_category'],
    )
    scanner_col = choose_column(
        metadata.columns,
        ['Device', 'scanner', 'equipment'],
    )
    bbox_col = choose_column(
        metadata.columns,
        ['BBOX', 'bbox', 'bounding_box'],
    )

    print(
        'BUS-BRA column mapping:',
        {
            'image_id': image_col,
            'patient_id': patient_col,
            'side': side_col,
            'histology': histology_col,
            'pathology': pathology_col,
            'birads': birads_col,
            'scanner': scanner_col,
            'bbox': bbox_col,
        },
    )

    if image_col is None:
        raise RuntimeError(
            'BUS-BRA ID column was not detected.'
        )

    if patient_col is None:
        raise RuntimeError(
            'BUS-BRA Case/patient column was not detected.'
        )

    if side_col is None:
        raise RuntimeError(
            'BUS-BRA Side column was not detected. '
            'It is required by the official filename convention.'
        )

    print(
        'First metadata ID/Side pairs:',
        metadata[
            [image_col, patient_col, side_col]
        ].head(12).to_dict(orient='records'),
    )

    side_distribution = (
        metadata[side_col]
        .map(normalize_busbra_side)
        .value_counts(dropna=False)
        .to_dict()
    )
    print(
        'Normalized Side distribution:',
        side_distribution,
    )

    records = []
    unresolved = []

    for row_index, row in metadata.iterrows():
        official_id = clean_identifier(
            row[image_col]
        )
        patient_id = clean_identifier(
            row[patient_col]
        )
        raw_side = clean_identifier(
            row[side_col]
        )

        normalized_id = normalize_busbra_id(
            official_id
        )
        normalized_side = normalize_busbra_side(
            raw_side
        )

        if (
            not official_id
            or not patient_id
            or not normalized_id
            or not normalized_side
        ):
            unresolved.append(
                {
                    'row_index': row_index,
                    'ID': official_id,
                    'Case': patient_id,
                    'Side': raw_side,
                    'normalized_id': normalized_id,
                    'normalized_side': normalized_side,
                    'reason': 'missing_id_case_or_side',
                }
            )
            continue

        image_path, image_match_method = lookup_busbra_asset(
            image_by_id_side,
            image_by_id,
            normalized_id,
            normalized_side,
        )

        mask_path, mask_match_method = lookup_busbra_asset(
            mask_by_id_side,
            mask_by_id,
            normalized_id,
            normalized_side,
        )

        if image_path is None or mask_path is None:
            unresolved.append(
                {
                    'row_index': row_index,
                    'ID': official_id,
                    'Case': patient_id,
                    'Side': raw_side,
                    'normalized_id': normalized_id,
                    'normalized_side': normalized_side,
                    'image_match_method': image_match_method,
                    'mask_match_method': mask_match_method,
                    'image_candidates_for_id': ';'.join(
                        path.name
                        for path in image_by_id.get(
                            normalized_id,
                            [],
                        )
                    ),
                    'mask_candidates_for_id': ';'.join(
                        path.name
                        for path in mask_by_id.get(
                            normalized_id,
                            [],
                        )
                    ),
                    'reason': (
                        f'image={image_match_method};'
                        f'mask={mask_match_method}'
                    ),
                }
            )
            continue

        records.append(
            {
                'dataset': 'bus_bra',
                'patient_id': patient_id,
                'case_id': (
                    f'{normalized_id}-{normalized_side}'
                ),
                'image_id': official_id,
                'side': raw_side,
                'side_normalized': normalized_side,
                'image_path': str(image_path),
                'mask_path': str(mask_path),
                'image_match_method': image_match_method,
                'mask_match_method': mask_match_method,
                'histology': (
                    ''
                    if histology_col is None
                    else str(row[histology_col]).strip()
                ),
                'pathology': (
                    ''
                    if pathology_col is None
                    else str(row[pathology_col]).strip()
                ),
                'birads': (
                    ''
                    if birads_col is None
                    else str(row[birads_col]).strip()
                ),
                'scanner': (
                    ''
                    if scanner_col is None
                    else str(row[scanner_col]).strip()
                ),
                'bbox_official': (
                    ''
                    if bbox_col is None
                    else str(row[bbox_col]).strip()
                ),
            }
        )

    unresolved_frame = pd.DataFrame(
        unresolved
    )
    unresolved_path = (
        PROJECT_ROOT
        / 'busbra_unresolved_image_mask_pairs.csv'
    )
    unresolved_frame.to_csv(
        unresolved_path,
        index=False,
    )

    frame = pd.DataFrame(records)

    print(
        'Resolved rows before duplicate checks:',
        len(frame),
    )
    print(
        'Unresolved metadata rows:',
        len(unresolved_frame),
    )
    print(
        'Parsed filename audit:',
        parsed_assets_path,
    )
    print(
        'Unresolved report:',
        unresolved_path,
    )

    if not unresolved_frame.empty:
        display(
            unresolved_frame.head(30)
        )

    if frame.empty:
        raise RuntimeError(
            'No BUS-BRA image/mask pairs were assembled. '
            'Inspect the parsed-filename and unresolved reports.'
        )

    frame = frame.drop_duplicates(
        [
            'patient_id',
            'case_id',
            'image_path',
            'mask_path',
        ]
    ).reset_index(drop=True)

    duplicate_case_ids = int(
        frame['case_id'].duplicated().sum()
    )
    duplicate_images = int(
        frame['image_path'].duplicated().sum()
    )
    duplicate_masks = int(
        frame['mask_path'].duplicated().sum()
    )

    print('BUS-BRA records assembled:', len(frame))
    print(
        'BUS-BRA unique patients:',
        frame['patient_id'].nunique(),
    )
    print(
        'Duplicate case IDs:',
        duplicate_case_ids,
    )
    print(
        'Duplicate image paths:',
        duplicate_images,
    )
    print(
        'Duplicate mask paths:',
        duplicate_masks,
    )

    if duplicate_case_ids > 0:
        raise RuntimeError(
            f'BUS-BRA contains {duplicate_case_ids} duplicate '
            'normalized (ID, Side) case IDs.'
        )

    if duplicate_images > 0 or duplicate_masks > 0:
        raise RuntimeError(
            'BUS-BRA image/mask paths are not one-to-one.'
        )

    if len(frame) != EXPECTED_BUSBRA_IMAGES:
        raise RuntimeError(
            'BUS-BRA pair count does not match the official release: '
            f'{len(frame)} assembled versus '
            f'{EXPECTED_BUSBRA_IMAGES} expected. '
            f'Inspect {unresolved_path} and '
            f'{parsed_assets_path}.'
        )

    patient_count = frame['patient_id'].nunique()

    if patient_count != EXPECTED_BUSBRA_PATIENTS:
        raise RuntimeError(
            'BUS-BRA patient count does not match the official release: '
            f'{patient_count} assembled versus '
            f'{EXPECTED_BUSBRA_PATIENTS} expected.'
        )

    print('BUS-BRA schema and pairing audit: PASS')
    return frame

busbra_records = build_busbra_records(
    BUSBRA_EXTRACTED
)

display(busbra_records.head())
display(
    busbra_records[
        [
            'image_id',
            'patient_id',
            'side',
            'case_id',
            'pathology',
            'birads',
            'scanner',
            'image_path',
            'mask_path',
        ]
    ].head(10)
)

In [ ]:
def clean_stratum(value: str) -> str:
    value = str(value).strip().lower()
    return value if value and value != 'nan' else 'unknown'

def make_patient_split(records: pd.DataFrame) -> pd.DataFrame:
    patient_table = (
        records.groupby('patient_id', as_index=False)
        .agg(
            pathology=('pathology', lambda x: clean_stratum(x.mode().iloc[0]) if not x.mode().empty else 'unknown'),
            scanner=('scanner', lambda x: clean_stratum(x.mode().iloc[0]) if not x.mode().empty else 'unknown'),
        )
    )
    patient_table['stratum'] = patient_table['pathology'] + '|' + patient_table['scanner']

    counts = patient_table['stratum'].value_counts()
    rare = set(counts[counts < 4].index)
    patient_table.loc[patient_table['stratum'].isin(rare), 'stratum'] = patient_table['pathology']

    counts = patient_table['stratum'].value_counts()
    rare = set(counts[counts < 3].index)
    patient_table.loc[patient_table['stratum'].isin(rare), 'stratum'] = 'all'

    stratify = patient_table['stratum'] if patient_table['stratum'].value_counts().min() >= 2 else None
    train, temp = train_test_split(
        patient_table,
        test_size=VAL_FRACTION + TEST_FRACTION,
        random_state=RANDOM_SEED,
        stratify=stratify,
    )

    temp_val_fraction = VAL_FRACTION / (VAL_FRACTION + TEST_FRACTION)
    temp_stratify = temp['stratum'] if temp['stratum'].value_counts().min() >= 2 else None
    validation, test = train_test_split(
        temp,
        train_size=temp_val_fraction,
        random_state=RANDOM_SEED,
        stratify=temp_stratify,
    )

    assignments = pd.concat([
        train.assign(split='train'),
        validation.assign(split='validation'),
        test.assign(split='test'),
    ], ignore_index=True)

    if assignments.patient_id.duplicated().any():
        raise RuntimeError('Duplicate patient assignment.')
    return assignments[['patient_id', 'split']]

split_assignments = make_patient_split(busbra_records)
busbra_records = busbra_records.merge(split_assignments, on='patient_id', how='left')
print(busbra_records.groupby('split').size())
print(busbra_records.groupby('split').patient_id.nunique())

In [ ]:

def normalize_header(name: str) -> str:
    return re.sub(
        r'[^a-z0-9]+',
        '',
        str(name).strip().lower(),
    )

def resolve_column(
    columns: Iterable[str],
    aliases: Iterable[str],
    required: bool = False,
) -> Optional[str]:
    normalized_map = {
        normalize_header(column): column
        for column in columns
    }

    for alias in aliases:
        key = normalize_header(alias)
        if key in normalized_map:
            return normalized_map[key]

    for alias in aliases:
        key = normalize_header(alias)
        for normalized, original in normalized_map.items():
            if key == normalized:
                return original

    if required:
        raise RuntimeError(
            f'Required BrEaST column not found. '
            f'Accepted aliases: {list(aliases)}. '
            f'Available columns: {list(columns)}'
        )

    return None

def locate_breast_asset(
    root: Path,
    filename: str,
) -> Optional[Path]:
    filename = str(filename).strip()

    if not filename:
        return None

    normalized_filename = filename.replace('\\', '/')
    basename = Path(normalized_filename).name

    exact = [
        path
        for path in root.rglob('*')
        if path.is_file() and path.name == basename
    ]
    if len(exact) == 1:
        return exact[0]

    case_insensitive = [
        path
        for path in root.rglob('*')
        if path.is_file()
        and path.name.lower() == basename.lower()
    ]
    if len(case_insensitive) == 1:
        return case_insensitive[0]

    target_stem = Path(basename).stem.lower()
    stem_matches = [
        path
        for path in root.rglob('*')
        if path.is_file()
        and path.stem.lower() == target_stem
    ]
    if len(stem_matches) == 1:
        return stem_matches[0]

    return None

def split_multi_filename(value) -> List[str]:
    """
    Gère les séparateurs observables dans les cellules XLSX :
    &, ;, virgule, saut de ligne ou pipe.
    """
    if value is None:
        return []

    text = str(value).strip()

    if (
        not text
        or text.lower() in {'nan', 'none', 'na', 'n/a'}
    ):
        return []

    parts = re.split(
        r'\s*(?:&|;|\||\n|,\s*(?=[A-Za-z0-9]))\s*',
        text,
    )

    return [
        part.strip()
        for part in parts
        if part.strip()
    ]

def safe_text(row: pd.Series, column: Optional[str]) -> str:
    if column is None:
        return ''

    value = row.get(column, '')

    if pd.isna(value):
        return ''

    return str(value).strip()

def build_breast_records(
    root: Path,
    xlsx_path: Path,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not xlsx_path.exists():
        raise FileNotFoundError(
            f'BrEaST clinical XLSX not found: {xlsx_path}'
        )

    clinical = pd.read_excel(
        xlsx_path,
        dtype=str,
        keep_default_na=False,
    )
    clinical.columns = [
        str(column).strip()
        for column in clinical.columns
    ]

    case_col = resolve_column(
        clinical.columns,
        [
            'CaseID',
            'Case_ID',
            'case_id',
            'patient_id',
            'PatientID',
        ],
        required=True,
    )
    image_col = resolve_column(
        clinical.columns,
        [
            'Image_filename',
            'ImageFileName',
            'image_file',
            'image',
        ],
        required=True,
    )
    tumor_mask_col = resolve_column(
        clinical.columns,
        [
            'Mask_tumor_filename',
            'Tumor_mask_filename',
            'MaskTumorFilename',
            'tumor_mask',
        ],
        required=True,
    )

    other_mask_col = resolve_column(
        clinical.columns,
        [
            'Mask_other_filename',
            'Other_mask_filename',
        ],
    )
    classification_col = resolve_column(
        clinical.columns,
        [
            'Classification',
            'Class',
            'Pathology',
        ],
    )
    diagnosis_col = resolve_column(
        clinical.columns,
        [
            'Diagnosis',
            'Histology',
        ],
    )
    birads_col = resolve_column(
        clinical.columns,
        [
            'BIRADS',
            'BI_RADS',
            'BI-RADS',
        ],
    )
    verification_col = resolve_column(
        clinical.columns,
        [
            'Verification',
            'Verification_method',
        ],
    )
    pixel_size_col = resolve_column(
        clinical.columns,
        [
            'Pixel_size',
            'PixelSize',
        ],
    )
    age_col = resolve_column(
        clinical.columns,
        ['Age'],
    )
    tissue_col = resolve_column(
        clinical.columns,
        [
            'Tissue_composition',
            'TissueComposition',
        ],
    )

    print(
        'BrEaST column mapping:',
        {
            'patient_id': case_col,
            'image_filename': image_col,
            'tumor_mask_filename': tumor_mask_col,
            'other_mask_filename': other_mask_col,
            'classification': classification_col,
            'diagnosis': diagnosis_col,
            'birads': birads_col,
            'verification': verification_col,
            'pixel_size': pixel_size_col,
            'age': age_col,
            'tissue_composition': tissue_col,
        },
    )

    records = []
    excluded = []

    for row_index, row in clinical.iterrows():
        patient_id = safe_text(row, case_col)
        image_name = safe_text(row, image_col)

        if not patient_id:
            excluded.append(
                {
                    'dataset': 'breast',
                    'row_index': row_index,
                    'patient_id': '',
                    'case_id': '',
                    'reason': 'missing_case_id',
                }
            )
            continue

        image_path = locate_breast_asset(
            root,
            image_name,
        )

        tumor_mask_names = split_multi_filename(
            safe_text(row, tumor_mask_col)
        )

        if not tumor_mask_names:
            excluded.append(
                {
                    'dataset': 'breast',
                    'row_index': row_index,
                    'patient_id': patient_id,
                    'case_id': patient_id,
                    'image_filename': image_name,
                    'reason': 'normal_or_missing_tumor_mask',
                }
            )
            continue

        if image_path is None:
            excluded.append(
                {
                    'dataset': 'breast',
                    'row_index': row_index,
                    'patient_id': patient_id,
                    'case_id': patient_id,
                    'image_filename': image_name,
                    'reason': f'image_not_found:{image_name}',
                }
            )
            continue

        for lesion_index, mask_name in enumerate(
            tumor_mask_names,
            start=1,
        ):
            mask_path = locate_breast_asset(
                root,
                mask_name,
            )

            if mask_path is None:
                excluded.append(
                    {
                        'dataset': 'breast',
                        'row_index': row_index,
                        'patient_id': patient_id,
                        'case_id': patient_id,
                        'image_filename': image_name,
                        'mask_filename': mask_name,
                        'reason': f'mask_not_found:{mask_name}',
                    }
                )
                continue

            case_id = (
                f'{patient_id}_lesion{lesion_index:02d}'
            )

            records.append(
                {
                    'dataset': 'breast',
                    'patient_id': patient_id,
                    'case_id': case_id,
                    'image_path': str(image_path),
                    'mask_path': str(mask_path),
                    'image_filename': image_name,
                    'mask_filename': mask_name,
                    'pathology': safe_text(
                        row,
                        classification_col,
                    ),
                    'diagnosis': safe_text(
                        row,
                        diagnosis_col,
                    ),
                    'birads': safe_text(
                        row,
                        birads_col,
                    ),
                    'verification': safe_text(
                        row,
                        verification_col,
                    ),
                    'pixel_size': safe_text(
                        row,
                        pixel_size_col,
                    ),
                    'age': safe_text(
                        row,
                        age_col,
                    ),
                    'tissue_composition': safe_text(
                        row,
                        tissue_col,
                    ),
                    'mask_other_filename': safe_text(
                        row,
                        other_mask_col,
                    ),
                    'split': 'external',
                    'lesion_index': lesion_index,
                }
            )

    records_frame = pd.DataFrame(records)
    excluded_frame = pd.DataFrame(excluded)

    if records_frame.empty:
        raise RuntimeError(
            'No BrEaST lesion records were assembled. '
            'Inspect the image/mask filenames and the excluded report.'
        )

    duplicate_cases = int(
        records_frame['case_id'].duplicated().sum()
    )

    duplicate_pairs = int(
        records_frame.duplicated(
            ['patient_id', 'image_path', 'mask_path']
        ).sum()
    )

    print('BrEaST clinical rows:', len(clinical))
    print(
        'BrEaST lesion records assembled:',
        len(records_frame),
    )
    print(
        'BrEaST unique patients with lesions:',
        records_frame['patient_id'].nunique(),
    )
    print(
        'BrEaST excluded/normal rows:',
        len(excluded_frame),
    )
    print(
        'BrEaST duplicate case IDs:',
        duplicate_cases,
    )
    print(
        'BrEaST duplicate image-mask pairs:',
        duplicate_pairs,
    )

    if duplicate_cases > 0:
        raise RuntimeError(
            f'BrEaST contains {duplicate_cases} duplicate case IDs.'
        )

    if duplicate_pairs > 0:
        raise RuntimeError(
            f'BrEaST contains {duplicate_pairs} duplicate image-mask pairs.'
        )

    return records_frame, excluded_frame

breast_records, breast_excluded = build_breast_records(
    BREAST_EXTRACTED,
    DOWNLOAD_ROOT / 'BrEaST-clinical-data.xlsx',
)

breast_excluded_path = (
    PROJECT_ROOT
    / 'breast_excluded_or_unresolved.csv'
)
breast_excluded.to_csv(
    breast_excluded_path,
    index=False,
)

print(
    'BrEaST exclusions report:',
    breast_excluded_path,
)

display(breast_records.head())
display(
    breast_records[
        [
            'patient_id',
            'case_id',
            'pathology',
            'diagnosis',
            'birads',
            'image_filename',
            'mask_filename',
        ]
    ].head(10)
)

In [ ]:
NPZ_ROOT = PREPARED_ROOT / 'npz'
NPZ_ROOT.mkdir(parents=True, exist_ok=True)

def prepare_record(row: pd.Series) -> Tuple[Optional[Dict], Optional[Dict]]:
    try:
        image = read_gray(Path(row.image_path))
        mask = read_binary_mask(Path(row.mask_path))

        if image.shape != mask.shape:
            mask = cv2.resize(
                mask,
                (image.shape[1], image.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )
            mask = (mask > 0).astype(np.uint8)

        if mask.sum() == 0:
            raise ValueError('empty_mask')

        original_ratio = float(mask.mean())
        image_norm = normalize_ultrasound(image)
        crop_image, crop_mask, crop_meta = crop_square_with_context(image_norm, mask)
        crop_image, crop_mask = resize_pair(crop_image, crop_mask)

        if crop_mask.sum() == 0:
            raise ValueError('empty_mask_after_resize')

        dataset = str(row.dataset)
        split = str(row.split)
        sample_id = f'{dataset}_{row.case_id}'
        safe_sample_id = re.sub(r'[^A-Za-z0-9_.-]+', '_', sample_id)

        output_dir = NPZ_ROOT / dataset / split
        output_dir.mkdir(parents=True, exist_ok=True)
        npz_path = output_dir / f'{safe_sample_id}.npz'

        np.savez_compressed(
            npz_path,
            image=crop_image.astype(np.float32),
            mask=crop_mask.astype(np.uint8),
        )

        result = {
            'sample_id': safe_sample_id,
            'dataset': dataset,
            'split': split,
            'patient_id': str(row.patient_id),
            'global_patient_id': f'{dataset}::{str(row.patient_id)}',
            'case_id': str(row.case_id),
            'image_source_path': str(row.image_path),
            'mask_source_path': str(row.mask_path),
            'npz_path': str(npz_path),
            'pathology': str(getattr(row, 'pathology', '')),
            'birads': str(getattr(row, 'birads', '')),
            'scanner': str(getattr(row, 'scanner', '')),
            'original_h': int(image.shape[0]),
            'original_w': int(image.shape[1]),
            'lesion_ratio_original': original_ratio,
            'lesion_ratio_crop': float(crop_mask.mean()),
            'oracle_crop_flag': True,
            **crop_meta,
            'image_hash': array_hash(crop_image),
            'mask_hash': array_hash(crop_mask),
            'pair_hash': hashlib.sha256(
                np.ascontiguousarray(crop_image).tobytes()
                + np.ascontiguousarray(crop_mask).tobytes()
            ).hexdigest(),
        }
        return result, None
    except Exception as error:
        failure = {
            'dataset': str(getattr(row, 'dataset', '')),
            'patient_id': str(getattr(row, 'patient_id', '')),
            'case_id': str(getattr(row, 'case_id', '')),
            'image_path': str(getattr(row, 'image_path', '')),
            'mask_path': str(getattr(row, 'mask_path', '')),
            'reason': f'{type(error).__name__}:{error}',
        }
        return None, failure

all_source_records = pd.concat(
    [busbra_records, breast_records],
    ignore_index=True,
    sort=False,
)

prepared, failures = [], []
if RUN_PREPARATION:
    for _, row in tqdm(all_source_records.iterrows(), total=len(all_source_records)):
        result, failure = prepare_record(row)
        if result is not None:
            prepared.append(result)
        if failure is not None:
            failures.append(failure)

manifest = pd.DataFrame(prepared)
failures_df = pd.concat(
    [breast_excluded, pd.DataFrame(failures)],
    ignore_index=True,
    sort=False,
)

manifest_path = PREPARED_ROOT / 'roi_us_manifest.csv'
failures_path = PREPARED_ROOT / 'failures.csv'
manifest.to_csv(manifest_path, index=False)
failures_df.to_csv(failures_path, index=False)

print('Prepared:', len(manifest))
print('Failures/exclusions:', len(failures_df))
print(manifest.groupby(['dataset', 'split']).size())

In [ ]:
#     global_patient_id = dataset + "::" + patient_id

def validate_npz(path: Path) -> Optional[str]:
    try:
        with np.load(path) as data:
            if set(data.files) != {'image', 'mask'}:
                return f'keys={data.files}'

            image = data['image']
            mask = data['mask']

        if image.shape != (CROP_SIZE, CROP_SIZE):
            return f'image_shape={image.shape}'

        if mask.shape != (CROP_SIZE, CROP_SIZE):
            return f'mask_shape={mask.shape}'

        if image.dtype != np.float32:
            return f'image_dtype={image.dtype}'

        if mask.dtype != np.uint8:
            return f'mask_dtype={mask.dtype}'

        if not np.isfinite(image).all():
            return 'image_non_finite'

        if image.min() < 0 or image.max() > 1:
            return (
                f'image_range=[{float(image.min())},'
                f'{float(image.max())}]'
            )

        if not set(np.unique(mask)).issubset({0, 1}):
            return 'mask_not_binary'

        if mask.sum() == 0:
            return 'empty_mask'

        return None

    except Exception as error:
        return f'{type(error).__name__}:{error}'



manifest_path = PREPARED_ROOT / 'roi_us_manifest.csv'

if 'manifest' not in globals() or manifest is None or len(manifest) == 0:
    if not manifest_path.exists():
        raise FileNotFoundError(
            f'Manifest not found: {manifest_path}'
        )
    manifest = pd.read_csv(
        manifest_path,
        dtype={
            'dataset': str,
            'split': str,
            'patient_id': str,
            'case_id': str,
        },
    )

required_manifest_columns = {
    'dataset',
    'split',
    'patient_id',
    'case_id',
    'npz_path',
}

missing_manifest_columns = (
    required_manifest_columns
    - set(manifest.columns)
)

if missing_manifest_columns:
    raise RuntimeError(
        'Manifest missing required columns: '
        f'{sorted(missing_manifest_columns)}'
    )

manifest['dataset'] = (
    manifest['dataset']
    .astype(str)
    .str.strip()
    .str.lower()
)

manifest['split'] = (
    manifest['split']
    .astype(str)
    .str.strip()
    .str.lower()
)

manifest['patient_id'] = (
    manifest['patient_id']
    .astype(str)
    .str.strip()
)

manifest['global_patient_id'] = (
    manifest['dataset']
    + '::'
    + manifest['patient_id']
)

manifest.to_csv(
    manifest_path,
    index=False,
)



validation_rows = []

for path_value in tqdm(
    manifest['npz_path'].astype(str).tolist(),
    desc='Validating NPZ',
):
    path = Path(path_value)
    reason = validate_npz(path)

    validation_rows.append(
        {
            'npz_path': str(path),
            'ok': reason is None,
            'reason': reason or '',
        }
    )

npz_validation = pd.DataFrame(
    validation_rows
)

audit_dir = PREPARED_ROOT / 'audit'
audit_dir.mkdir(
    parents=True,
    exist_ok=True,
)

npz_validation.to_csv(
    audit_dir / 'npz_full_validation.csv',
    index=False,
)



expected_structure = {
    'bus_bra': {'train', 'validation', 'test'},
    'breast': {'external'},
}

observed_structure = {
    dataset: set(group['split'].unique())
    for dataset, group in manifest.groupby('dataset')
}

unexpected_dataset_rows = manifest[
    ~manifest['dataset'].isin(
        expected_structure.keys()
    )
].copy()

busbra_wrong_split_rows = manifest[
    (manifest['dataset'] == 'bus_bra')
    & ~manifest['split'].isin(
        ['train', 'validation', 'test']
    )
].copy()

breast_wrong_split_rows = manifest[
    (manifest['dataset'] == 'breast')
    & (manifest['split'] != 'external')
].copy()

missing_required_dataset = [
    dataset
    for dataset in expected_structure
    if dataset not in observed_structure
]



busbra_patient_sets = {
    split: set(
        manifest.loc[
            (manifest['dataset'] == 'bus_bra')
            & (manifest['split'] == split),
            'global_patient_id',
        ]
    )
    for split in [
        'train',
        'validation',
        'test',
    ]
}

internal_leakage_sets = {
    'train_validation': (
        busbra_patient_sets['train']
        & busbra_patient_sets['validation']
    ),
    'train_test': (
        busbra_patient_sets['train']
        & busbra_patient_sets['test']
    ),
    'validation_test': (
        busbra_patient_sets['validation']
        & busbra_patient_sets['test']
    ),
}

internal_leakage = {
    key: len(value)
    for key, value in internal_leakage_sets.items()
}

external_partition_ok = (
    len(breast_wrong_split_rows) == 0
    and len(
        manifest[
            (manifest['dataset'] == 'bus_bra')
            & (manifest['split'] == 'external')
        ]
    ) == 0
)



missing_npz_mask = ~manifest['npz_path'].map(
    lambda value: Path(str(value)).exists()
)

missing_npz = int(
    missing_npz_mask.sum()
)

invalid_npz = int(
    (~npz_validation['ok']).sum()
)

duplicate_sample_ids = int(
    manifest['sample_id'].duplicated().sum()
) if 'sample_id' in manifest.columns else 0

duplicate_npz_paths = int(
    manifest['npz_path'].duplicated().sum()
)

duplicate_pairs = int(
    manifest['pair_hash'].duplicated().sum()
) if 'pair_hash' in manifest.columns else 0

empty_patient_ids = int(
    (
        manifest['patient_id'].isna()
        | manifest['patient_id'].eq('')
        | manifest['patient_id'].eq('nan')
    ).sum()
)

empty_case_ids = int(
    (
        manifest['case_id'].isna()
        | manifest['case_id'].astype(str).str.strip().eq('')
    ).sum()
)

busbra_patient_count = int(
    manifest.loc[
        manifest['dataset'] == 'bus_bra',
        'patient_id',
    ].nunique()
)

breast_patient_count = int(
    manifest.loc[
        manifest['dataset'] == 'breast',
        'patient_id',
    ].nunique()
)

busbra_crop_count = int(
    (manifest['dataset'] == 'bus_bra').sum()
)

breast_crop_count = int(
    (manifest['dataset'] == 'breast').sum()
)



blocking_reasons = []

if len(manifest) == 0:
    blocking_reasons.append(
        'manifest_empty'
    )

if invalid_npz > 0:
    blocking_reasons.append(
        f'invalid_npz={invalid_npz}'
    )

if missing_npz > 0:
    blocking_reasons.append(
        f'missing_npz={missing_npz}'
    )

if any(
    value > 0
    for value in internal_leakage.values()
):
    blocking_reasons.append(
        f'busbra_internal_patient_leakage={internal_leakage}'
    )

if not external_partition_ok:
    blocking_reasons.append(
        'breast_is_not_fully_external'
    )

if len(unexpected_dataset_rows) > 0:
    blocking_reasons.append(
        f'unexpected_dataset_rows={len(unexpected_dataset_rows)}'
    )

if len(busbra_wrong_split_rows) > 0:
    blocking_reasons.append(
        f'busbra_wrong_split_rows={len(busbra_wrong_split_rows)}'
    )

if len(breast_wrong_split_rows) > 0:
    blocking_reasons.append(
        f'breast_wrong_split_rows={len(breast_wrong_split_rows)}'
    )

if missing_required_dataset:
    blocking_reasons.append(
        f'missing_required_dataset={missing_required_dataset}'
    )

if duplicate_sample_ids > 0:
    blocking_reasons.append(
        f'duplicate_sample_ids={duplicate_sample_ids}'
    )

if duplicate_npz_paths > 0:
    blocking_reasons.append(
        f'duplicate_npz_paths={duplicate_npz_paths}'
    )

if empty_patient_ids > 0:
    blocking_reasons.append(
        f'empty_patient_ids={empty_patient_ids}'
    )

if empty_case_ids > 0:
    blocking_reasons.append(
        f'empty_case_ids={empty_case_ids}'
    )

if busbra_patient_count == 0:
    blocking_reasons.append(
        'busbra_has_no_patients'
    )

if breast_patient_count == 0:
    blocking_reasons.append(
        'breast_has_no_patients'
    )

go = len(blocking_reasons) == 0



for name, frame in {
    'unexpected_dataset_rows.csv': unexpected_dataset_rows,
    'busbra_wrong_split_rows.csv': busbra_wrong_split_rows,
    'breast_wrong_split_rows.csv': breast_wrong_split_rows,
    'missing_npz_rows.csv': manifest.loc[missing_npz_mask],
}.items():
    frame.to_csv(
        audit_dir / name,
        index=False,
    )

leakage_rows = []
for comparison, patient_ids in internal_leakage_sets.items():
    for patient_id in sorted(patient_ids):
        leakage_rows.append(
            {
                'comparison': comparison,
                'global_patient_id': patient_id,
            }
        )

pd.DataFrame(
    leakage_rows,
    columns=[
        'comparison',
        'global_patient_id',
    ],
).to_csv(
    audit_dir / 'busbra_internal_patient_leakage.csv',
    index=False,
)

audit_summary = {
    'manifest_rows': int(len(manifest)),
    'npz_validated': int(len(npz_validation)),
    'invalid_npz': invalid_npz,
    'missing_npz': missing_npz,
    'duplicate_sample_ids': duplicate_sample_ids,
    'duplicate_npz_paths': duplicate_npz_paths,
    'duplicate_pair_hashes_nonblocking': duplicate_pairs,
    'empty_patient_ids': empty_patient_ids,
    'empty_case_ids': empty_case_ids,
    'busbra_crop_count': busbra_crop_count,
    'busbra_patient_count': busbra_patient_count,
    'breast_crop_count': breast_crop_count,
    'breast_patient_count': breast_patient_count,
    'busbra_internal_patient_leakage': internal_leakage,
    'external_partition_ok': bool(
        external_partition_ok
    ),
    'observed_structure': {
        key: sorted(value)
        for key, value in observed_structure.items()
    },
    'counts_by_dataset_split': (
        manifest
        .groupby(
            ['dataset', 'split']
        )
        .size()
        .rename('n')
        .reset_index()
        .to_dict(orient='records')
    ),
    'patients_by_dataset_split': (
        manifest
        .groupby(
            ['dataset', 'split']
        )['patient_id']
        .nunique()
        .rename('n_patients')
        .reset_index()
        .to_dict(orient='records')
    ),
    'blocking_reasons': blocking_reasons,
    'verdict': 'GO' if go else 'NO-GO',
}

(PREPARED_ROOT / 'audit_summary.json').write_text(
    json.dumps(
        audit_summary,
        indent=2,
    ),
    encoding='utf-8',
)

counts_table = (
    manifest
    .groupby(
        ['dataset', 'split']
    )
    .size()
    .rename('n')
    .reset_index()
)

patients_table = (
    manifest
    .groupby(
        ['dataset', 'split']
    )['patient_id']
    .nunique()
    .rename('n_patients')
    .reset_index()
)

audit_md = [
    '# Audit ROI_US_Crops_256_v1',
    '',
    f'**Verdict : {audit_summary["verdict"]}**',
    '',
    '## Résumé',
    '',
    f'- Manifest rows : {len(manifest)}',
    f'- NPZ invalides : {invalid_npz}',
    f'- NPZ manquants : {missing_npz}',
    f'- BUS-BRA crops : {busbra_crop_count}',
    f'- BUS-BRA patientes : {busbra_patient_count}',
    f'- BrEaST crops : {breast_crop_count}',
    f'- BrEaST patientes : {breast_patient_count}',
    f'- Fuites internes BUS-BRA : {internal_leakage}',
    f'- BrEaST entièrement externe : {external_partition_ok}',
    f'- Doublons pair_hash non bloquants : {duplicate_pairs}',
    '',
    '## Raisons bloquantes',
    '',
    (
        '- Aucune'
        if not blocking_reasons
        else '\n'.join(
            f'- {reason}'
            for reason in blocking_reasons
        )
    ),
    '',
    '## Crops par dataset et split',
    '',
    counts_table.to_markdown(
        index=False
    ),
    '',
    '## Patientes par dataset et split',
    '',
    patients_table.to_markdown(
        index=False
    ),
    '',
    '## Règles',
    '',
    '- BUS-BRA = développement.',
    '- BrEaST = externe scellé.',
    '- Les patient_id sont namespacés par dataset.',
    '- Cas normaux BrEaST sans masque exclus du résultat principal.',
    '- Crop oracle avec facteur de contexte 1,5.',
]

(PREPARED_ROOT / 'AUDIT_ROI_US_256_v1.md').write_text(
    '\n'.join(audit_md),
    encoding='utf-8',
)

print(
    json.dumps(
        audit_summary,
        indent=2,
    )
)

if not go:
    print('\nAUDIT NO-GO — blocking reasons:')
    for reason in blocking_reasons:
        print(' -', reason)

    raise AssertionError(
        'Audit NO-GO. Blocking reasons: '
        + ' | '.join(blocking_reasons)
    )

print('\nAUDIT GO — dataset preparation is valid.')

In [ ]:
import matplotlib.pyplot as plt

sample = manifest.groupby(['dataset', 'split'], group_keys=False).apply(
    lambda frame: frame.sample(min(3, len(frame)), random_state=RANDOM_SEED)
).reset_index(drop=True)

fig, axes = plt.subplots(len(sample), 3, figsize=(11, 3 * len(sample)))
if len(sample) == 1:
    axes = np.array([axes])

for row_index, row in sample.iterrows():
    with np.load(row.npz_path) as data:
        image = data['image']
        mask = data['mask']

    overlay = np.stack([image, image, image], axis=-1)
    overlay[..., 0] = np.maximum(overlay[..., 0], mask * 0.9)

    axes[row_index, 0].imshow(image, cmap='gray')
    axes[row_index, 0].set_title(f'{row.dataset} / {row.split}')
    axes[row_index, 1].imshow(mask, cmap='gray')
    axes[row_index, 1].set_title('Mask')
    axes[row_index, 2].imshow(overlay)
    axes[row_index, 2].set_title('Overlay')
    for axis in axes[row_index]:
        axis.axis('off')

plt.tight_layout()
figure_path = PREPARED_ROOT / 'audit' / 'roi_us_overlay_check.png'
plt.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print(figure_path)

In [ ]:
light_root = PROJECT_ROOT / 'ROI_US_Crops_256_v1_AUDIT_LIGHT'
if light_root.exists():
    shutil.rmtree(light_root)
light_root.mkdir(parents=True)

for filename in [
    'roi_us_manifest.csv',
    'failures.csv',
    'audit_summary.json',
    'AUDIT_ROI_US_256_v1.md',
]:
    shutil.copy2(PREPARED_ROOT / filename, light_root / filename)

shutil.copytree(
    PREPARED_ROOT / 'audit',
    light_root / 'audit',
    dirs_exist_ok=True,
)

light_zip = shutil.make_archive(
    str(PROJECT_ROOT / 'ROI_US_Crops_256_v1_AUDIT_LIGHT'),
    'zip',
    root_dir=light_root,
)

full_zip = None
if CREATE_FULL_ZIP:
    full_zip = shutil.make_archive(
        str(PROJECT_ROOT / 'ROI_US_Crops_256_v1_FULL'),
        'zip',
        root_dir=PREPARED_ROOT,
    )

print('Prepared dataset:', PREPARED_ROOT)
print('Light audit ZIP:', light_zip)
print('Full dataset ZIP:', full_zip)